# 偏置系统核心概念详解

## 什么是偏置（Bias）？

作者认为，**偏置**不是随机的倾向性，而是**有目的的引导机制**。它将问题的先验知识、领域经验、约束条件转化为可计算的策略，引导优化过程朝更有希望的方向搜索。

### 偏置的本质

- **知识注入** - 将专家经验、历史数据、问题特性转化为偏置
- **方向引导** - 告诉算法哪些区域更有希望
- **动态调整** - 根据搜索进展自适应调整策略
- **约束处理** - 优雅地处理各种约束条件

In [ ]:
# 导入必要的库
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 环境准备完成")
print(f"当前工作目录: {os.getcwd()}")

## 1. 理解偏置基类（BiasBase）

所有偏置都继承自 `BiasBase`，它定义了偏置系统的核心接口：

- `apply()` - 应用偏置到种群
- `initialize()` - 初始化偏置
- `update()` - 更新偏置参数

In [ ]:
# 创建一个简单的偏置示例
class SimpleBias:
    """简单的偏置示例 - 向零点偏置"""
    
    def __init__(self, bias_strength=0.1):
        self.bias_strength = bias_strength
        print(f"创建简单偏置，强度: {bias_strength}")
    
    def apply(self, population):
        """应用偏置"""
        biased_pop = []
        target = np.zeros(len(population[0]))  # 偏向原点
        
        for ind in population:
            # 向目标点移动一小步
            new_ind = np.array(ind) + self.bias_strength * (target - np.array(ind))
            biased_pop.append(new_ind)
        return biased_pop

# 测试偏置
print("测试简单偏置：")
bias = SimpleBias(bias_strength=0.2)

# 创建测试种群
test_pop = np.random.randn(5, 2)  # 5个个体，2维
print(f"\n原始种群（前3个）: \n{test_pop[:3]}")

# 应用偏置
biased_pop = bias.apply(test_pop)
print(f"\n偏置后种群（前3个）: \n{np.array(biased_pop)[:3]}")

## 2. 偏置的分类

偏置系统采用三层架构：

### 算法级偏置
- **选择偏置** - 影响哪些个体被选入下一代
- **变异偏置** - 引导变异方向
- **交叉偏置** - 控制交叉策略

### 领域级偏置
- **约束偏置** - 处理约束条件
- **偏好偏置** - 融入决策者偏好
- **启发式偏置** - 利用问题特性

### 自适应偏置
- **动态调整** - 根据搜索过程调整
- **学习机制** - 从历史中学习

In [ ]:
# 创建约束偏置示例
class ConstraintBias:
    """约束偏置示例"""
    
    def __init__(self, bounds=(-1, 1)):
        self.lower_bound, self.upper_bound = bounds
        
    def apply(self, population):
        """应用约束偏置 - 将解限制在边界内"""
        constrained_pop = []
        for ind in population:
            # 使用裁剪（clip）将值限制在边界内
            constrained_ind = np.clip(ind, self.lower_bound, self.upper_bound)
            constrained_pop.append(constrained_ind)
        return constrained_pop

# 演示约束偏置
print("演示约束偏置：")
constraint_bias = ConstraintBias(bounds=(-2, 2))

# 创建超出边界的种群
unconstrained_pop = np.array([
    [3.0, 1.0],   # 超出上界
    [-3.0, 0.5],  # 超出下界
    [0.5, 2.5],   # 部分超出
    [0.0, 0.0]    # 在界内
])

print(f"\n原始种群：")
for i, ind in enumerate(unconstrained_pop):
    print(f"  个体{i}: {ind}")

# 应用约束偏置
constrained_pop = constraint_bias.apply(unconstrained_pop)

print(f"\n约束后种群：")
for i, ind in enumerate(constrained_pop):
    print(f"  个体{i}: {ind}")

## 3. 偏置效果可视化

In [ ]:
# 可视化偏置效果
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 生成随机种群
np.random.seed(42)
pop_size = 100
population = np.random.randn(pop_size, 2) * 3  # 均值为0，标准差为3

# 应用向零点偏置
zero_bias = SimpleBias(bias_strength=0.3)
biased_pop = zero_bias.apply(population)

# 左图：原始种群
ax1 = axes[0]
ax1.scatter(population[:, 0], population[:, 1], alpha=0.6, s=30)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('原始种群分布')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([-10, 10])
ax1.set_ylim([-10, 10])

# 右图：偏置后种群
ax2 = axes[1]
ax2.scatter(np.array(biased_pop)[:, 0], np.array(biased_pop)[:, 1], 
           alpha=0.6, s=30, c='red')
ax2.scatter(0, 0, s=100, c='black', marker='*', label='目标点')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('向零点偏置后分布')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim([-10, 10])
ax2.set_ylim([-10, 10])

plt.tight_layout()
plt.show()

# 计算统计信息
original_mean_dist = np.mean(np.linalg.norm(population, axis=1))
biased_mean_dist = np.mean(np.linalg.norm(biased_pop, axis=1))

print(f"\n📊 偏置效果统计：")
print(f"  原始种群平均距离原点: {original_mean_dist:.3f}")
print(f"  偏置后种群平均距离原点: {biased_mean_dist:.3f}")
print(f"  收缩比例: {biased_mean_dist/original_mean_dist:.3f}")

## 4. 多种偏置组合

In [ ]:
# 创建偏置组合系统
class CompositeBias:
    """复合偏置 - 组合多个偏置"""
    
    def __init__(self, biases):
        self.biases = biases
        print(f"创建复合偏置，包含 {len(biases)} 个子偏置")
    
    def apply(self, population):
        """顺序应用所有偏置"""
        result = population.copy()
        for bias in self.biases:
            result = bias.apply(result)
        return result

# 创建局部搜索偏置
class LocalSearchBias:
    """局部搜索偏置 - 在当前解附近小幅扰动"""
    
    def __init__(self, step_size=0.1):
        self.step_size = step_size
    
    def apply(self, population):
        """应用局部搜索"""
        local_pop = []
        for ind in population:
            # 添加小幅随机扰动
            noise = np.random.normal(0, self.step_size, len(ind))
            local_ind = np.array(ind) + noise
            local_pop.append(local_ind)
        return local_pop

# 演示偏置组合
print("演示偏置组合：")

# 创建复合偏置
composite_bias = CompositeBias([
    ConstraintBias(bounds=(-5, 5)),  # 先应用约束
    SimpleBias(bias_strength=0.2),     # 然后向零点偏置
    LocalSearchBias(step_size=0.05)    # 最后局部搜索
])

# 创建测试种群（包含超出边界的点）
test_pop = np.array([
    [8.0, 3.0],   # 超出边界
    [-6.0, 2.0],  # 超出边界
    [1.0, 1.0],   # 在界内
    [0.5, -0.5],  # 在界内
    [2.0, 2.0]    # 在界内
])

print(f"\n原始种群：")
for i, ind in enumerate(test_pop):
    print(f"  个体{i}: {ind}")

# 应用复合偏置
result_pop = composite_bias.apply(test_pop)

print(f"\n复合偏置后种群：")
for i, ind in enumerate(result_pop):
    print(f"  个体{i}: {ind}")

# 可视化复合偏置效果
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 原始种群
ax1 = axes[0]
ax1.scatter(test_pop[:, 0], test_pop[:, 1], s=100, alpha=0.7)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('原始种群')
ax1.grid(True, alpha=0.3)

# 约束偏置后
constraint_bias = ConstraintBias(bounds=(-5, 5))
constrained_pop = constraint_bias.apply(test_pop)
ax2 = axes[1]
ax2.scatter(np.array(constrained_pop)[:, 0], np.array(constrained_pop)[:, 1], 
           s=100, alpha=0.7, c='orange')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('约束偏置后')
ax2.grid(True, alpha=0.3)

# 最终复合偏置结果
ax3 = axes[2]
ax3.scatter(np.array(result_pop)[:, 0], np.array(result_pop)[:, 1], 
           s=100, alpha=0.7, c='red')
ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_title('复合偏置后')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 总结

### 核心要点

1. **偏置的本质** - 有目的的引导，而非随机倾向
2. **三层架构** - 算法级、领域级、自适应偏置
3. **组合使用** - 多个偏置可以组合使用
4. **可配置性** - 通过参数控制偏置行为

### 使用建议

- 根据问题特点选择合适的偏置
- 调整偏置强度，避免过度偏置导致早熟
- 尝试组合不同类型的偏置
- 监控偏置效果，及时调整策略

偏置系统是NSGABLACK框架的核心特色，掌握偏置的概念和使用方法将显著提升优化算法的能力。